# 📈 Reporte Ejecutivo: Simulación Estratégica (Monte Carlo)
---

## 1. Fundamento de la Simulación
**¿Por qué usamos Simulaciones de Monte Carlo?**  
En el entorno turístico, una predicción de "una sola línea" es peligrosa porque asume que el futuro es 100% predecible. Al utilizar Monte Carlo, le pedimos a la Inteligencia Artificial que simule **500 futuros posibles** basados en el caos matemático de la historia. Esto nos permite descubrir el verdadero **Riesgo (VaR - Value at Risk)**: no solo adivinar qué pasará, sino entender **qué tan mal se pueden poner las cosas** para que el negocio esté blindado.

## 2. Inclusión de Crisis Históricas
**¿El modelo toma en cuenta pandemias, crisis económicas y huelgas? SÍ.**  
El núcleo matemático de este modelo (`Prophet`) fue alimentado con los siguientes **Shocks Estructurales**:

*   **La Crisis Subprime (2008-2009) y H1N1:** Inyectada a través del fuerte pico en la **Tasa de Desempleo de USA** (Nuestro mayor mercado proxy).
*   **Pandemia COVID-19 (2020-2021):** Introducida como un 'Shock' explícito masivo en la matriz de demanda para medir la elasticidad del colapso.
*   **Huelga Nacional (2018):** Introducido para medir interrupciones sociales de corto plazo en Costa Rica.

El algoritmo estudió cómo reaccionó la ocupación ante estos infiernos, y usa ese aprendizaje para "castigar" la simulación del 2026-2027 si ocurren eventos de riesgo.

In [1]:
import pandas as pd
import numpy as np
import sys
import os
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as pyo

# Localizar rutas base
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..')) if 'notebooks' in os.getcwd() else os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'data', 'merged', 'master_tourism_data.csv')
REPORTS_PATH = os.path.join(BASE_DIR, 'reports')

# Asegurar carga del motor de inteligencia
sys.path.append(os.path.join(BASE_DIR, 'src'))
from forecasting_engine import run_pro_montecarlo_forecast

pyo.init_notebook_mode(connected=True)
pio.renderers.default = 'notebook_connected'
TEMPLATE = 'plotly_dark'

print(f'✅ Contexto histórico cargado. Inicializando Motor Estocástico...')

✅ Contexto histórico cargado. Inicializando Motor Estocástico...


---
## 3. Ejecutando 500 Futuros Alternativos
La siguiente celda inyecta la crisis y calcula 500 caminos de probabilidad independientes para generar una matriz de estrés.

In [2]:
df_raw = pd.read_csv(DATA_PATH)
df_raw['DATE'] = pd.to_datetime(df_raw['DATE'])

res = run_pro_montecarlo_forecast(
    df_raw, 
    recovery_months=12, 
    econ_multiplier=1.0, 
    n_samples=500
)

print('✅ 500 Simulaciones generadas con éxito.')

✅ 500 Simulaciones generadas con éxito.


---
## 📊 El Gráfico Oficial de Simulación (Spaghetti Plot)
A diferencia de una zona sombreada genérica, este gráfico expone de fondo **una muestra de 100 escenarios caóticos individuales (líneas finas)** calculados por el motor. 

*   **Línea Púrpura Gruesa:** El escenario base (El futuro más probable).
*   **Red de Líneas Traslúcidas:** Los 100 universos paralelos de riesgo. Si las líneas se esparcen mucho, denotan extrema volatilidad.
*   **Línea Ámbar:** El piso de riesgo (Peor escenario P05).

In [3]:
import numpy as np
import plotly.graph_objects as go
fig = go.Figure()

future_idx = [i for i, d in enumerate(res['dates']) if d > res['actual_dates'][-1]]
future_dates = np.array(res['dates'])[future_idx]
last_date = res['actual_dates'][-1]
last_y = res['actual_y'][-1]
f_dates_pinned = [last_date] + list(future_dates)

p95_pinned = [last_y] + list(np.array(res['p95'])[future_idx])
p05_pinned = [last_y] + list(np.array(res['p05'])[future_idx])
med_pinned = [last_y] + list(np.array(res['median'])[future_idx])

fig.add_trace(go.Scatter(x=f_dates_pinned, y=p95_pinned, mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=f_dates_pinned, y=p05_pinned, fill='tonexty', fillcolor='rgba(139, 92, 246, 0.25)', mode='lines', line=dict(width=0), name='Banda de Riesgo (90% Confianza)'))
fig.add_trace(go.Scatter(x=f_dates_pinned, y=p95_pinned, mode='lines', line=dict(color='rgba(139, 92, 246, 0.8)', width=2, dash='dot'), name='Escenario Optimista (P95)'))
fig.add_trace(go.Scatter(x=f_dates_pinned, y=p05_pinned, mode='lines', line=dict(color='#ff3366', width=2, dash='dot'), name='Escenario Crítico (P05)'))
fig.add_trace(go.Scatter(x=f_dates_pinned, y=med_pinned, mode='lines', line=dict(color='#10b981', width=4), name='Proyección Central Base'))
fig.add_trace(go.Scatter(x=res['actual_dates'], y=res['actual_y'], mode='lines', line=dict(color='#ffffff', width=2.5), name='Historia Ocupación Real'))

fig.update_layout(
    template='plotly_dark', height=650,
    title=dict(text='<b>PROYECCIÓN DE OCUPACIÓN Y MAPA DE RIESGOS (2025-2027)</b><br><sup><i>Basado en 500 iteraciones Monte Carlo evaluando choques de seguridad y volatilidad económica.</i></sup>', font=dict(color='#ffffff', size=20)),
    yaxis=dict(title='Ocupación Hotelera (%)', range=[0, 100], gridcolor='rgba(255,255,255,0.05)', zeroline=False, tickfont=dict(size=14)),
    xaxis=dict(title='', gridcolor='rgba(255,255,255,0.05)', zeroline=False, tickfont=dict(size=14)),
    plot_bgcolor='#0f172a', paper_bgcolor='#0f172a', hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(size=13), bgcolor='rgba(0,0,0,0)'),
    margin=dict(l=60, r=40, t=110, b=40)
)
fig.show()

---
## 💾 Cierre y Exportación de Límites
Los datos han sido validados a través de 500 iteraciones estocásticas y exportados para uso en dashboards ejecutivos (Excel/BI).

In [4]:
df_final = pd.DataFrame({'DATE': res['dates'], 'Median': res['median'], 'P05': res['p05'], 'P95': res['p95']})
export_file = os.path.join(REPORTS_PATH, 'final_mc_forecast_2027.csv')
df_final[df_final['DATE'] > res['actual_dates'][-1]].to_csv(export_file, index=False)
print(f'🛡️ Datos de Riesgo Validados y Guardados en: {export_file}')

🛡️ Datos de Riesgo Validados y Guardados en: C:\Users\franc\Downloads\guanacaste-tourism-forecaster\guanacaste-tourism-forecaster\reports\final_mc_forecast_2027.csv
